# Setup

In [1]:
import numpy as np
import pandas as pd 

In [2]:
# Read csv into a DataFrame
prices =\
(
    pd
    .read_csv(
        "prices.csv",
        parse_dates = ["Date"],
        index_col = "Date",
    )
)
prices

,SP500,NASDAQCOM,DJIA,WTI,USD/EUR,VIX
Date,,,,,,
2018-01-01,NaN,NaN,NaN,NaN,1.200495,NaN
2018-01-02,2695.810059,7006.899902,24824.009766,60.369999,1.201158,9.77
2018-01-03,2713.060059,7065.529785,24922.679688,61.630001,1.206345,9.15
2018-01-04,2723.989990,7077.910156,25075.130859,62.009998,1.201043,9.22
2018-01-05,2743.149902,7136.560059,25295.869141,61.439999,1.206884,9.22
...,...,...,...,...,...,...
2024-12-25,NaN,NaN,NaN,NaN,1.040258,NaN
2024-12-26,6037.589844,20020.359375,43325.800781,69.620003,1.039955,14.73
2024-12-27,5970.839844,19722.029297,42992.210938,70.599998,1.042318,15.95


In [3]:
macro =\
(
    pd
    .read_csv(
        "macro.csv",
        parse_dates = ["Date"],
        index_col = "Date",
    )
)
macro

,y10,y2,spread
Date,,,
2018-01-01,NaN,NaN,NaN
2018-01-02,2.46,1.92,0.54
2018-01-03,2.44,1.94,0.50
2018-01-04,2.46,1.96,0.50
2018-01-05,2.47,1.96,0.51
...,...,...,...
2024-12-25,NaN,NaN,NaN
2024-12-26,4.58,4.30,0.28
2024-12-27,4.62,4.31,0.31


In [4]:
from lets_plot import * 
LetsPlot.setup_html()

# <mark>geom_line</mark> for time series (singular instrument)

In [5]:
prices_LONG =\
(
    prices
    .reset_index()
    .melt(id_vars = "Date",
          var_name = "Instrument",
          value_name = "Price"        
    )
)
prices_LONG

,Date,Instrument,Price
0,2018-01-01,SP500,NaN
1,2018-01-02,SP500,2695.810059
2,2018-01-03,SP500,2713.060059
3,2018-01-04,SP500,2723.989990
4,2018-01-05,SP500,2743.149902
...,...,...,...
10957,2024-12-25,VIX,NaN
10958,2024-12-26,VIX,14.730000
10959,2024-12-27,VIX,15.950000
10960,2024-12-30,VIX,17.400000


In [6]:
prices_LONG["Instrument"].unique()

array(['SP500', 'NASDAQCOM', 'DJIA', 'WTI', 'USD/EUR', 'VIX'],
      dtype=object)

In [7]:
(
    ggplot(prices_LONG.query("Instrument == 'SP500'"),
           aes(x = "Date",
               y = "Price")
           )
    + geom_line(color = "red",
                alpha = 0.30
               )
)

In [48]:
(
    ggplot(prices.reset_index(),
           aes(x = "Date", 
               y = "SP500")
            )
    + geom_line(color = "grey")
    + geom_vline(xintercept = pd.Timestamp("2020-03-23"),
                 color = "red"
                )
    + geom_text(x = pd.Timestamp("2020-03-23"),
                y = 5000,
                label = "Covid Low")
)

# geom_line for time series (multiple instruments)

In [8]:
normalized_price =\
(
    prices.loc["2018-01-02" : "2024-12-31",["SP500", "NASDAQCOM", "DJIA"]]
    .div(prices.loc["2018-01-02" : "2024-12-31",["SP500", "NASDAQCOM", "DJIA"]]
        .iloc[0]
        )
    .mul(100)
)
normalized_price

,SP500,NASDAQCOM,DJIA
Date,,,
2018-01-02,100.000000,100.000000,100.000000
2018-01-03,100.639882,100.836745,100.397478
2018-01-04,101.045323,101.013433,101.011606
2018-01-05,101.756053,101.850464,101.900819
2018-01-08,101.925206,102.147743,101.848977
...,...,...,...
2024-12-25,NaN,NaN,NaN
2024-12-26,223.961990,285.723496,174.531839
2024-12-27,221.485925,281.465835,173.188020


In [9]:
normalized_price_LONG =\
(
    normalized_price
    .reset_index()
    .melt(id_vars = "Date", 
          var_name = "Instrument",
          value_name = "Normalized price")
)
normalized_price_LONG

,Date,Instrument,Normalized price
0,2018-01-02,SP500,100.000000
1,2018-01-03,SP500,100.639882
2,2018-01-04,SP500,101.045323
3,2018-01-05,SP500,101.756053
4,2018-01-08,SP500,101.925206
...,...,...,...
5473,2024-12-25,DJIA,NaN
5474,2024-12-26,DJIA,174.531839
5475,2024-12-27,DJIA,173.188020
5476,2024-12-30,DJIA,171.502231


In [10]:
(
    ggplot(normalized_price_LONG,
           aes(x = "Date",
               y = "Normalized price")
            )
    + geom_line(aes(color = "Instrument")
                )
    #+ scale_y_log10()
    + theme(legend_position = "bottom")
)

In [11]:
(
    ggplot(normalized_price_LONG.query("Instrument in ['SP500', 'NASDAQCOM']"),
           aes(x = "Date",
               y = "Normalized price")
            )
    + geom_line(aes(color = "Instrument")
                )
    + scale_y_log10()
)

# geom_line for rolling statistics

In [12]:
rollvol =\
(
    prices["SP500"]
    .pct_change()
    .rolling(21)
    .std()
    .mul(np.sqrt(252))
    .mul(100)                # to percentage terms
    .to_frame()
)
rollvol

/var/folders/qn/f0b3bn_x4m70c5_zp1rw74080000gn/T/ipykernel_1322/3433624241.py:4: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()


,SP500
Date,
2018-01-01,NaN
2018-01-02,NaN
2018-01-03,NaN
2018-01-04,NaN
2018-01-05,NaN
...,...
2024-12-25,13.439128
2024-12-26,13.363652
2024-12-27,13.935793


In [13]:
(
    ggplot(rollvol.reset_index(),
           aes(x = "Date",
               y= "SP500")
            )
    + geom_line()
    + geom_hline(yintercept = rollvol["SP500"].mean(), linetype = "dashed", color = "red")
    + labs(y = "Vol (%)",
           title = "Rolling 21-day Annualized Vol - SP500 (%)")
)

# <mark>geom_point</mark> for relationships (basic scatter plot)


In [14]:
rets =\
(
    prices
    .pct_change()
    .dropna()
    .mul(100)
)
rets

/var/folders/qn/f0b3bn_x4m70c5_zp1rw74080000gn/T/ipykernel_1322/3574833154.py:4: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()


,SP500,NASDAQCOM,DJIA,WTI,USD/EUR,VIX
Date,,,,,,
2018-01-03,0.639882,0.836745,0.397478,2.087133,0.431876,-6.345965
2018-01-04,0.402864,0.175222,0.611697,0.616578,-0.439584,0.765034
2018-01-05,0.703377,0.828633,0.880308,-0.919206,0.486369,0.000000
2018-01-08,0.166234,0.291878,-0.050874,0.472007,-0.260004,3.253798
2018-01-09,0.130293,0.086483,0.406600,1.992547,-0.556629,5.882347
...,...,...,...,...,...,...
2024-12-25,0.000000,0.000000,0.000000,0.000000,-0.031206,0.000000
2024-12-26,-0.040566,-0.053774,0.066447,-0.684730,-0.029107,3.223539
2024-12-27,-1.105574,-1.490133,-0.769957,1.407635,0.227218,8.282419


In [15]:
(
    ggplot(rets, 
           aes(x = "WTI", 
               y = "SP500")
            )
    + geom_point(alpha = 0.25)
    + scale_x_log10()
    + scale_y_log10()
    + geom_smooth(method = "lm",
                  color = "red",      # color OUTSIDE aes() sets one color for every mark
    #              se = False
                )
)

# geom_point for relationships (scatter plot, by year)


In [16]:
# Add new columns into "rets" DataFrame to mark the year (as a number) and the year (as a string)

rets_with_years =\
(
    rets
    .assign(year = rets.index.year,
            year_str = rets.index.year.astype(str)
            )
)
rets_with_years

,SP500,NASDAQCOM,DJIA,WTI,USD/EUR,VIX,year,year_str
Date,,,,,,,,
2018-01-03,0.639882,0.836745,0.397478,2.087133,0.431876,-6.345965,2018,2018
2018-01-04,0.402864,0.175222,0.611697,0.616578,-0.439584,0.765034,2018,2018
2018-01-05,0.703377,0.828633,0.880308,-0.919206,0.486369,0.000000,2018,2018
2018-01-08,0.166234,0.291878,-0.050874,0.472007,-0.260004,3.253798,2018,2018
2018-01-09,0.130293,0.086483,0.406600,1.992547,-0.556629,5.882347,2018,2018
...,...,...,...,...,...,...,...,...
2024-12-25,0.000000,0.000000,0.000000,0.000000,-0.031206,0.000000,2024,2024
2024-12-26,-0.040566,-0.053774,0.066447,-0.684730,-0.029107,3.223539,2024,2024
2024-12-27,-1.105574,-1.490133,-0.769957,1.407635,0.227218,8.282419,2024,2024


In [18]:
# Using year (as a number)

(
    ggplot(rets_with_years, 
           aes(x = "SP500", 
               y = "VIX")
            )
    + geom_point(aes(color = "year"),      
                 alpha = 0.25
                 )
)

In [19]:
# Using year (as a string)

(
    ggplot(rets_with_years, 
           aes(x = "SP500", 
               y = "VIX")
            )
    + geom_point(aes(color = "year_str",
                 alpha = 0.25)
                 )
)

ERR [SpecTransformBackendUtil] : Internal error: ClassCastException : class kotlin.Double cannot be cast to class kotlin.String


In [20]:
# `as_discrete()` converts numerical variable into categorical variable in situ

(
    ggplot(rets_with_years, 
           aes(x = "SP500", 
               y = "VIX")
            )
    + geom_point(aes(color = as_discrete("year")),
                 alpha = 0.25
                 )
)

In [21]:
# customization of the categorical variable with `scale_color_manual`

(
    ggplot(rets_with_years, 
           aes(x = "SP500", 
               y = "VIX")
            )
    + geom_point(aes(color = "year_str"),
                 alpha = 0.25
                 )
    + scale_color_manual(values = {"2018": "red", 
                                   "2019": "red",
                                   "2020": "blue",
                                   "2021": "green", 
                                   "2021": "green",
                                   "2022": "orange", 
                                   "2023": "orange", 
                                   "2024": "orange"}
                        )
)

In [22]:
(
    ggplot(rets.reset_index(), 
           aes(x = "Date",
               y = "SP500", 
               color = "VIX"))
    + geom_point()
    + scale_color_gradient(low = "#cccccc", high = "#b30000")
)

# geom_point for relationships (mean-variance plot)


In [24]:
# Create DataFrame for mean-variance summary
mean_variance =\
(
    prices[prices.columns[:-2]]
    .pct_change()
    .dropna()
    .agg(["std", "mean"])
    .T
    .rename(columns = {"std": "vol", "mean": "ret"})
)

mean_variance["vol"] =\
(
    mean_variance["vol"]
    .mul(252 ** 0.5)
)

mean_variance["ret"] =\
(
    mean_variance["ret"]
    .mul(252)
)

mean_variance["sharpe"] =\
(
    mean_variance["ret"] / mean_variance["vol"] 
)

mean_variance.round(3)

/var/folders/qn/f0b3bn_x4m70c5_zp1rw74080000gn/T/ipykernel_1322/4180764894.py:5: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()


,vol,ret,sharpe
SP500,0.194,0.127,0.654
NASDAQCOM,0.233,0.167,0.719
DJIA,0.191,0.093,0.486
WTI,1.321,-0.377,-0.285


In [25]:
(
    ggplot(mean_variance.reset_index(),      # add one column named "index" for use in labels
           aes(x = "vol",
               y = "ret")
           )
    + geom_point(aes(color = "index",
                 size = "sharpe")
                 )
    + geom_text(aes(label = "index"),
                nudge_y = 0.00, 
                nudge_x = 0.25
                )
    + scale_y_continuous(limits = [-0.5,0.2],
                         breaks = np.arange(-0.5, 0.25, 0.05)
                         )
    + scale_x_continuous(limits = [0,1.6],
                             breaks = np.arange(0,1.7,0.1)
                             )
)

# <mark>geom_smooth</mark> for a trend

In [ ]:
# Create DataFrame combining macro data 

panel =\
(
    prices[["SP500"]]
    .join(macro[["y10"]]
          )
    .dropna()
)

# Create a new column for year (as a string)

panel =\
(
    panel
    .assign(year = panel.index.year.astype(str)
            )
)

panel

,SP500,y10,year
Date,,,
2018-01-02,2695.810059,2.46,2018
2018-01-03,2713.060059,2.44,2018
2018-01-04,2723.989990,2.46,2018
2018-01-05,2743.149902,2.47,2018
2018-01-08,2747.709961,2.49,2018
...,...,...,...
2024-12-24,6040.040039,4.59,2024
2024-12-26,6037.589844,4.58,2024
2024-12-27,5970.839844,4.62,2024


In [27]:
(
    ggplot(panel,
           aes(x = "y10",
               y = "SP500")
            )
    + geom_point(alpha = 0.15)
    + geom_smooth(aes(color = "year"),           # color INSIDE aes() maps a column to color
                  method = "lm",
                  se = False
                  )
)

# <mark>geom_bar(stat="identity")</mark> for totals

In [29]:
totals =\
(
    (1 + rets/100)
    .prod()
    .sub(1)
    .mul(100)
    .rename("Total_Percent_Change")
    .reset_index()
)
totals

,index,Total_Percent_Change
0,SP500,118.176717
1,NASDAQCOM,175.596759
2,DJIA,71.383347
3,WTI,18.800733
4,USD/EUR,-13.365663
5,VIX,77.584438


In [ ]:
(
    ggplot(totals, 
           aes(x = "index",
               y = "Total_Percent_Change")
            )
    + geom_bar(aes(fill = np.where(totals["Total_Percent_Change"] < 0, "negative", "positive")),
               stat = "identity",
               )
    + scale_fill_manual(values = ["green", "red"],
                        name = "Performance"
                        )
    + coord_flip()
)

# geom_histogram or <mark>geom_density</mark> for distributions

In [32]:
(
    ggplot(rets,
           aes("SP500"))
    + geom_histogram(bins = 50)
)

In [34]:
(
    ggplot(rets/100,
           aes(x = "SP500")
           )
    + geom_density(kernel = "gaussian")
)

In [35]:
# Plot multiple assets on one chart - first get a LONG format DataFrame

rets_LONG =\
(
    rets
    .reset_index()
    .melt(id_vars = "Date")
)
rets_LONG["value"] =\
(
    rets_LONG["value"]
    .divide(100)
)

rets_LONG

,Date,variable,value
0,2018-01-03,SP500,0.006399
1,2018-01-04,SP500,0.004029
2,2018-01-05,SP500,0.007034
3,2018-01-08,SP500,0.001662
4,2018-01-09,SP500,0.001303
...,...,...,...
10945,2024-12-25,VIX,0.000000
10946,2024-12-26,VIX,0.032235
10947,2024-12-27,VIX,0.082824
10948,2024-12-30,VIX,0.090909


In [36]:
(
    ggplot(rets_LONG,
           aes(x = "value")
           )
    + geom_density(aes(fill = "variable"),
                   kernel = "gaussian",
                   alpha = 0.30)
    + scale_x_continuous(limits = [-0.2, 0.2],
                         breaks = np.arange(-0.2, 0.3, 0.1)
                         )
)

### or stack lines manually

In [44]:
(
    ggplot(rets)
    + geom_density(aes (x = "SP500"), 
                   fill = "blue",
                   alpha = 0.20
                   )
    + geom_density(aes (x = "NASDAQCOM"), 
                   fill = "red", 
                   alpha = 0.20
                   )
)

# <mark>geom_area</mark> for cumulative growth / drawdown

In [37]:
growth =\
(
    (1 + rets[["SP500"]]/100)
    .cumprod()
    .rename(columns = {"SP500" : "Growth_of_1_Dollar"})
    .reset_index()
)
growth

,Date,Growth_of_1_Dollar
0,2018-01-03,1.006399
1,2018-01-04,1.010453
2,2018-01-05,1.017561
3,2018-01-08,1.019252
4,2018-01-09,1.020580
...,...,...
1820,2024-12-25,2.240529
1821,2024-12-26,2.239620
1822,2024-12-27,2.214859
1823,2024-12-30,2.191156


In [38]:
(
    ggplot(growth,
           aes(x = "Date",
               y = "Growth_of_1_Dollar")
            )
    + geom_area(fill = "green",
                alpha = 0.30)
)

In [ ]:
growth_path =\
(
    prices["SP500"]
    .pct_change()
    .add(1)
    .cumprod()
)
growth_path

/var/folders/qn/f0b3bn_x4m70c5_zp1rw74080000gn/T/ipykernel_1322/3511277696.py:4: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()


Date
2018-01-01         NaN
2018-01-02         NaN
2018-01-03    1.006399
2018-01-04    1.010453
2018-01-05    1.017561
                ...   
2024-12-25    2.240529
2024-12-26    2.239620
2024-12-27    2.214859
2024-12-30    2.191156
2024-12-31    2.181767
Name: SP500, Length: 1827, dtype: float64

In [ ]:
DrawDown =\
(
    growth_path
    .div(growth_path.cummax()
        )
    .sub(1)
    .rename("drawdown")
    .reset_index()
)
DrawDown

,Date,drawdown
0,2018-01-01,NaN
1,2018-01-02,NaN
2,2018-01-03,0.000000
3,2018-01-04,0.000000
4,2018-01-05,0.000000
...,...,...
1822,2024-12-25,-0.008248
1823,2024-12-26,-0.008650
1824,2024-12-27,-0.019610
1825,2024-12-30,-0.030102


In [ ]:
a = prices.loc['2022', 'SP500']

In [ ]:
(
    ggplot(DrawDown,
           aes(x = "Date",
               y = "drawdown")
            )
    + geom_area(fill = "red",
                alpha = 0.30)
    + labs(title = f"2022 inside-year returns: {a.iloc[-1]/ a.iloc[0] - 1:.1%}")
)

# <mark>geom_boxplot</mark> to compare distributions

In [39]:
rets_LONG

,Date,variable,value
0,2018-01-03,SP500,0.006399
1,2018-01-04,SP500,0.004029
2,2018-01-05,SP500,0.007034
3,2018-01-08,SP500,0.001662
4,2018-01-09,SP500,0.001303
...,...,...,...
10945,2024-12-25,VIX,0.000000
10946,2024-12-26,VIX,0.032235
10947,2024-12-27,VIX,0.082824
10948,2024-12-30,VIX,0.090909


In [40]:
(
    ggplot(rets_LONG.query("variable != 'VIX'"),
           aes(x = "variable",
               y = "value")
            )
    + geom_boxplot(aes(fill = "variable"))
    + scale_y_continuous(limits = [-0.3, 0.3],
                         breaks = np.arange(-0.3, 0.4, 0.1))
    + coord_flip()
)

# <mark>geom_tile</mark> for heatmaps (correlation)

In [ ]:
# Create correlation DataFrame

corr_LONG =\
(
    rets
    .corr()
    # .reset_index()
    # .melt(id_vars = "index",
    #       var_name = "X",
    #       value_name = "corr")
    # .rename(columns = {"index": "Y"})
)
corr_LONG.head()

,Y,X,corr
0,SP500,SP500,1.000000
1,NASDAQCOM,SP500,0.948184
2,DJIA,SP500,0.952076
3,WTI,SP500,0.133379
4,USD/EUR,SP500,0.030976
